# 07 - Scaling Survey

Efficient `N_PSR x N_CW` survey under realistic stochastic noise. Uses tiered diagnostics instead of the full recovery pipeline.

In [4]:
from pathlib import Path
import sys
import time
import copy
import importlib
import itertools

import jax
import jax.numpy as jnp
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.signal import find_peaks
from IPython.display import display

HERE = Path.cwd()
sys.path.insert(0, str((HERE / "../CW_lnL_check").resolve()))

import cw_helpers
importlib.reload(cw_helpers)

from cw_helpers import (
    build_disco_likelihood,
    build_fast_scan_likelihood,
    build_enterprise_pta,
    compute_mode_spacing,
    generate_injection_params,
    load_pulsars,
    make_distance_optimizer,
    scan_pulsar_distance,
    simulate,
)

# Survey grid
N_PSR_VALUES = [10, 20, 40, 60, 80]
N_CW_VALUES = [2, 4, 6, 8, 12, 16]
MAX_N_CW = max(N_CW_VALUES)

# Realistic stochastic setup
RNG_SEED = 12345
NOISE_SEED = 24680
LOG10_H = -12.0
LOG10_MC = None
STOCHASTIC_SCENARIO = "well_separated"
COMPONENTS = 14
VALIDATE_COMPONENTS = True
VALIDATION_N_PSR = 20
VALIDATION_N_CW = 8
INCLUDE_GWB = True
GWB_LOG10_A = -14.5
GWB_GAMMA = 13 / 3
INCLUDE_RN = True
RN_COMPONENTS = COMPONENTS

# Distance/prior setup
TRUTH_SIGMA_CLIP = 3.0
TRUTH_SIGMA_MULTIPLIER = 1.0
MODE_PRIOR_SIGMA_WIDTH = 3.0

# Tier controls
RUN_TIER_0 = True
RUN_TIER_1 = True
RUN_TIER_2 = True
TIER_2_CONTRAST_THRESHOLD = 2.0
TIER1_SCAN_POINTS = 1500
TIER1_N_PULSARS = 5
COORD_DESCENT_SWEEPS = 3
COORD_DESCENT_SCAN_POINTS = 1000
MAKE_PLOTS = True

# Runtime knobs
USE_FAST_LIKELIHOOD = True
OPT_FACTR = 1e7
TIER0_OPT_MAXITER = 20
FORCE_REBUILD = False

rng_master = np.random.default_rng(RNG_SEED)
np.random.seed(NOISE_SEED)
print(HERE)

/home/mattm/projects/HSYMT/lnL_distance_scans


## Shared Helpers

In [5]:
CW_LIBRARY = [
    dict(cos_gwtheta=0.30, gwphi=2.50, cos_inc=-0.20, phase0=1.00, psi=0.70, log10_h=-12.0, log10_mc=9.00, log10_fgw=-8.00),
    dict(cos_gwtheta=-0.50, gwphi=0.80, cos_inc=0.40, phase0=2.10, psi=1.30, log10_h=-12.0, log10_mc=9.00, log10_fgw=-7.80),
    dict(cos_gwtheta=0.05, gwphi=4.20, cos_inc=-0.65, phase0=0.35, psi=2.30, log10_h=-12.0, log10_mc=9.00, log10_fgw=-8.20),
    dict(cos_gwtheta=0.70, gwphi=5.40, cos_inc=0.10, phase0=1.70, psi=0.20, log10_h=-12.0, log10_mc=9.00, log10_fgw=-7.90),
    dict(cos_gwtheta=-0.10, gwphi=3.30, cos_inc=0.80, phase0=2.90, psi=1.80, log10_h=-12.0, log10_mc=9.00, log10_fgw=-8.10),
    dict(cos_gwtheta=0.45, gwphi=1.60, cos_inc=-0.35, phase0=0.80, psi=2.70, log10_h=-12.0, log10_mc=9.00, log10_fgw=-7.70),
    dict(cos_gwtheta=-0.75, gwphi=5.90, cos_inc=0.55, phase0=2.40, psi=0.45, log10_h=-12.0, log10_mc=9.00, log10_fgw=-8.35),
    dict(cos_gwtheta=0.18, gwphi=0.25, cos_inc=-0.85, phase0=1.35, psi=1.05, log10_h=-12.0, log10_mc=9.00, log10_fgw=-7.60),
    dict(cos_gwtheta=-0.32, gwphi=2.05, cos_inc=0.25, phase0=3.05, psi=2.05, log10_h=-12.0, log10_mc=9.00, log10_fgw=-8.45),
    dict(cos_gwtheta=0.88, gwphi=3.75, cos_inc=-0.05, phase0=0.15, psi=0.95, log10_h=-12.0, log10_mc=9.00, log10_fgw=-7.95),
    dict(cos_gwtheta=-0.62, gwphi=4.75, cos_inc=0.72, phase0=1.95, psi=2.85, log10_h=-12.0, log10_mc=9.00, log10_fgw=-8.25),
    dict(cos_gwtheta=0.58, gwphi=0.95, cos_inc=-0.48, phase0=2.75, psi=1.55, log10_h=-12.0, log10_mc=9.00, log10_fgw=-7.75),
    dict(cos_gwtheta=-0.18, gwphi=5.15, cos_inc=0.08, phase0=0.55, psi=0.10, log10_h=-12.0, log10_mc=9.00, log10_fgw=-8.05),
    dict(cos_gwtheta=0.02, gwphi=1.20, cos_inc=-0.70, phase0=2.25, psi=2.45, log10_h=-12.0, log10_mc=9.00, log10_fgw=-7.85),
    dict(cos_gwtheta=-0.88, gwphi=3.95, cos_inc=0.38, phase0=1.15, psi=1.75, log10_h=-12.0, log10_mc=9.00, log10_fgw=-8.30),
    dict(cos_gwtheta=0.36, gwphi=4.55, cos_inc=-0.18, phase0=2.55, psi=0.60, log10_h=-12.0, log10_mc=9.00, log10_fgw=-7.65),
]


def clipped_normal(mean, sigma, rng, clip):
    draw = rng.normal(mean, sigma)
    lo = np.maximum(0.01, mean - clip * sigma)
    hi = mean + clip * sigma
    return np.clip(draw, lo, hi)


def select_pulsars(n_psr):
    ent_all, disco_all = load_pulsars(None)
    pairs = sorted(zip(ent_all, disco_all), key=lambda pair: pair[1].pdist[1])
    selected = pairs[:n_psr]
    return [p[0] for p in selected], [p[1] for p in selected], len(disco_all)


def clone_psrs_with_distances(psrs, distances):
    clones = copy.deepcopy(psrs)
    for psr, dist in zip(clones, distances):
        sigma = float(psr.pdist[1])
        try:
            psr.pdist = (float(dist), sigma)
        except Exception:
            psr._pdist = (float(dist), sigma)
    return clones


def make_truth_distances(prior_mean, prior_sigma, rng):
    return clipped_normal(prior_mean, prior_sigma * TRUTH_SIGMA_MULTIPLIER, rng, TRUTH_SIGMA_CLIP)


def distance_key(psr):
    return f"{psr.name}_cw_p_dist"


def set_distances(values, param_keys, disco_psrs, distances):
    out = np.array(values, dtype=float).copy()
    for psr, dist in zip(disco_psrs, distances):
        out[param_keys.index(distance_key(psr))] = dist
    return out


def cw_list_from_enterprise_params(enterprise_params, cw_block_names):
    keys = ("cos_gwtheta", "gwphi", "cos_inc", "log10_mc", "log10_fgw", "log10_h", "phase0", "psi")
    return [{key: float(enterprise_params[f"{name}_{key}"]) for key in keys} for name in cw_block_names]


def silence_extra_cws(params, cw_block_names, active_n_cw):
    out = dict(params)
    for idx, name in enumerate(cw_block_names):
        if idx >= active_n_cw:
            out[f"{name}_log10_h"] = -50.0
    return out


def silence_base_values(base_values, param_keys, active_n_cw):
    out = np.array(base_values, dtype=float).copy()
    for cw_idx in range(active_n_cw, MAX_N_CW):
        suffix = "" if cw_idx == 0 else f"_{cw_idx + 1}"
        key = f"cw_log10_h{suffix}"
        if key in param_keys:
            out[param_keys.index(key)] = -50.0
    return out


def min_mode_spacings(disco_psrs, cw_params_list):
    out = []
    for psr in disco_psrs:
        vals = [compute_mode_spacing(cw["cos_gwtheta"], cw["gwphi"], cw["log10_fgw"], psr.pos) for cw in cw_params_list]
        out.append(float(np.nanmin(vals)))
    return np.array(out)


def score_recovery(distances, truth_dist, mode_spacings):
    err_modes = (np.asarray(distances) - truth_dist) / mode_spacings
    return dict(
        n_recovered=int(np.sum(np.abs(err_modes) < 0.5)),
        frac_recovered=float(np.mean(np.abs(err_modes) < 0.5)),
        median_abs_modes=float(np.nanmedian(np.abs(err_modes))),
        max_abs_modes=float(np.nanmax(np.abs(err_modes))),
    )


def top_local_maxima(x, y, k=200):
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    finite = np.isfinite(y)
    x, y = x[finite], y[finite]
    if len(x) == 0:
        return np.array([]), np.array([])
    if len(x) >= 3:
        idx = np.where(np.r_[False, (y[1:-1] >= y[:-2]) & (y[1:-1] >= y[2:]), False])[0]
    else:
        idx = np.array([], dtype=int)
    if len(idx) == 0:
        idx = np.array([int(np.nanargmax(y))])
    idx = idx[np.argsort(y[idx])[::-1]][:k]
    return x[idx], y[idx]


def diagnostic_indices(sigmas, mode_spacings, n=TIER1_N_PULSARS):
    hardness = sigmas / mode_spacings
    eligible = np.where(np.isfinite(hardness) & (hardness >= 5))[0]
    if len(eligible) == 0:
        eligible = np.arange(len(sigmas))
    order = eligible[np.argsort(hardness[eligible])]
    if len(order) <= n:
        return order.tolist()
    qs = np.linspace(0, len(order) - 1, n).round().astype(int)
    return order[qs].tolist()


def scan_metrics(case, i, context_distances, scan_points=TIER1_SCAN_POINTS):
    lo = max(0.01, case["prior_mean"][i] - MODE_PRIOR_SIGMA_WIDTH * case["sigmas"][i])
    hi = case["prior_mean"][i] + MODE_PRIOR_SIGMA_WIDTH * case["sigmas"][i]
    ctx = set_distances(case["base_values"], case["param_keys"], case["disco_psrs"], context_distances)
    vals, lls = scan_pulsar_distance(
        case["logl_fn"], ctx, case["param_keys"], distance_key(case["disco_psrs"][i]),
        lo, hi, n_points=scan_points, required_points=[case["truth_dist"][i], case["prior_mean"][i]],
        n_components=case["components"],
    )
    vals = np.asarray(vals, dtype=float)
    lls = np.asarray(lls, dtype=float)
    peaks_x, peaks_y = top_local_maxima(vals, lls, k=300)
    truth_i = int(np.argmin(np.abs(vals - case["truth_dist"][i])))
    truth_y = float(lls[truth_i])
    global_i = int(np.nanargmax(lls))
    truth_is_global = abs(vals[global_i] - case["truth_dist"][i]) < 0.5 * case["mode_spacings"][i] or truth_y >= float(np.nanmax(lls)) - 1e-6
    local = np.where(np.abs(vals - case["truth_dist"][i]) <= 0.5 * case["mode_spacings"][i])[0]
    valley = float(np.nanmin(lls[local])) if len(local) else float(np.nanmin(lls))
    n_degenerate = int(np.sum(peaks_y >= truth_y - 1.0)) if len(peaks_y) else 0
    return dict(
        contrast=float(truth_y - valley),
        n_degenerate=n_degenerate,
        truth_is_global=bool(truth_is_global),
        global_peak=float(vals[global_i]),
    )

## Build Likelihoods

In [6]:
case_cache = {}
pta_cache = {}
build_rows = []


def get_pta_bundle(n_psr, components=COMPONENTS):
    key = (int(n_psr), int(components))
    if key in pta_cache and not FORCE_REBUILD:
        return pta_cache[key]

    rng = np.random.default_rng(RNG_SEED)
    ent_psrs, disco_psrs, total_loaded = select_pulsars(n_psr)
    prior_mean = np.array([p.pdist[0] for p in disco_psrs], dtype=float)
    sigmas = np.array([p.pdist[1] for p in disco_psrs], dtype=float)
    truth_dist = make_truth_distances(prior_mean, sigmas, rng)
    ent_psrs_inj = clone_psrs_with_distances(ent_psrs, truth_dist)

    t0 = time.time()
    pta, cw_block_names, _ = build_enterprise_pta(
        ent_psrs_inj, MAX_N_CW, components=components,
        include_rn=INCLUDE_RN, rn_components=components,
    )
    params_full = generate_injection_params(
        pta, ent_psrs_inj, MAX_N_CW, cw_block_names,
        log10_h=LOG10_H,
        scenario=STOCHASTIC_SCENARIO,
        rng=rng,
        gwb_log10_A=GWB_LOG10_A,
        gwb_gamma=GWB_GAMMA,
        include_rn=INCLUDE_RN,
    )
    bundle = dict(
        pta=pta, cw_block_names=cw_block_names, params_full=params_full,
        ent_psrs=ent_psrs, disco_psrs=disco_psrs, total_loaded=total_loaded,
        prior_mean=prior_mean, sigmas=sigmas, truth_dist=truth_dist,
        components=components, pta_seconds=time.time() - t0,
    )
    pta_cache[key] = bundle
    print(f"built enterprise PTA N_PSR={n_psr}, components={components} in {bundle['pta_seconds']:.1f}s")
    return bundle


def build_case(n_psr, active_n_cw, components=COMPONENTS):
    key = (int(n_psr), int(active_n_cw), int(components))
    if key in case_cache and not FORCE_REBUILD:
        return case_cache[key]

    t0 = time.time()
    bundle = get_pta_bundle(n_psr, components)
    pta = bundle["pta"]
    cw_block_names = bundle["cw_block_names"]
    params_full = bundle["params_full"]
    ent_psrs = bundle["ent_psrs"]
    disco_psrs = bundle["disco_psrs"]
    prior_mean = bundle["prior_mean"]
    sigmas = bundle["sigmas"]
    truth_dist = bundle["truth_dist"]
    total_loaded = bundle["total_loaded"]

    params_active = silence_extra_cws(params_full, cw_block_names, active_n_cw)
    cw_params_all = cw_list_from_enterprise_params(params_active, cw_block_names)
    cw_params_active = cw_params_all[:active_n_cw]
    np.random.seed(NOISE_SEED)
    sim_resids = simulate(pta, params_active, sparse_cholesky=True)
    residual_map = {getattr(p, "name", p): y for p, y in zip(pta.pulsars, sim_resids)}

    logl_fn_disco, param_keys_disco, base_values_disco = build_disco_likelihood(
        disco_psrs, residual_map,
        num_cw=MAX_N_CW,
        enterprise_params=params_active,
        cw_block_names=cw_block_names,
        components=components,
        include_gwb=INCLUDE_GWB,
        include_rn=INCLUDE_RN,
        rn_components=components,
    )

    logl_fn, param_keys, base_values = logl_fn_disco, param_keys_disco, np.asarray(base_values_disco, dtype=float)
    if USE_FAST_LIKELIHOOD and INCLUDE_GWB:
        try:
            logl_fn_fast, param_keys_fast, base_values_fast = build_fast_scan_likelihood(
                disco_psrs, residual_map,
                num_cw=MAX_N_CW,
                enterprise_params=params_active,
                cw_block_names=cw_block_names,
                components=components,
                include_rn=INCLUDE_RN,
                rn_components=components,
            )
            if param_keys_fast == param_keys_disco:
                truth_probe = set_distances(base_values_fast, param_keys_fast, disco_psrs, truth_dist)
                delta = float(logl_fn_fast(jnp.asarray(truth_probe)) - logl_fn_disco(jnp.asarray(truth_probe)))
                if abs(delta) <= 1e-6:
                    logl_fn, param_keys, base_values = logl_fn_fast, param_keys_fast, np.asarray(base_values_fast, dtype=float)
                else:
                    print(f"  fast validation failed for {key}: delta={delta:.3e}; using discovery")
            else:
                print(f"  fast key mismatch for {key}; using discovery")
        except Exception as e:
            print(f"  fast likelihood failed for {key}: {e}; using discovery")

    base_values = silence_base_values(base_values, param_keys, active_n_cw)
    mode_spacings = min_mode_spacings(disco_psrs, cw_params_active)
    truth_values = set_distances(base_values, param_keys, disco_psrs, truth_dist)
    prior_values = set_distances(base_values, param_keys, disco_psrs, prior_mean)

    case = dict(
        key=key, n_psr=n_psr, active_n_cw=active_n_cw, components=components,
        ent_psrs=ent_psrs, disco_psrs=disco_psrs, total_loaded=total_loaded,
        prior_mean=prior_mean, sigmas=sigmas, truth_dist=truth_dist,
        cw_params_active=cw_params_active, logl_fn=logl_fn, param_keys=param_keys,
        base_values=base_values, truth_values=truth_values, prior_values=prior_values,
        mode_spacings=mode_spacings,
    )
    case_cache[key] = case
    build_seconds = time.time() - t0
    build_rows.append(dict(N_PSR=n_psr, N_CW=active_n_cw, components=components, seconds=build_seconds))
    print(f"built N_PSR={n_psr}, N_CW={active_n_cw}, components={components} in {build_seconds:.1f}s")
    return case


if RUN_TIER_0 or RUN_TIER_1 or RUN_TIER_2:
    for n_psr, n_cw in itertools.product(N_PSR_VALUES, N_CW_VALUES):
        build_case(n_psr, n_cw, COMPONENTS)

build_df = pd.DataFrame(build_rows)
display(build_df)

FeatherPulsar.read_feather: cannot find dmx in feather file /home/mattm/projects/HSYMT/data_products/B1855+09.feather.
FeatherPulsar.read_feather: cannot find _pdist in feather file /home/mattm/projects/HSYMT/data_products/B1855+09.feather.
FeatherPulsar.read_feather: cannot find dmx in feather file /home/mattm/projects/HSYMT/data_products/B1937+21.feather.
FeatherPulsar.read_feather: cannot find _pdist in feather file /home/mattm/projects/HSYMT/data_products/B1937+21.feather.
FeatherPulsar.read_feather: cannot find dmx in feather file /home/mattm/projects/HSYMT/data_products/B1953+29.feather.
FeatherPulsar.read_feather: cannot find _pdist in feather file /home/mattm/projects/HSYMT/data_products/B1953+29.feather.
FeatherPulsar.read_feather: cannot find dmx in feather file /home/mattm/projects/HSYMT/data_products/J0023+0923.feather.
FeatherPulsar.read_feather: cannot find _pdist in feather file /home/mattm/projects/HSYMT/data_products/J0023+0923.feather.
FeatherPulsar.read_feather: cann

Duplicate signal J0437-4715_cw from objects <Enterprise Signal object cw[cw2_cos_gwtheta, cw2_gwphi, cw2_cos_inc, cw2_log10_mc, cw2_log10_fgw, cw2_log10_h, cw2_phase0, cw2_psi, J0437-4715_cw_p_dist]> and <Enterprise Signal object cw[cw_cos_gwtheta, cw_gwphi, cw_cos_inc, cw_log10_mc, cw_log10_fgw, cw_log10_h, cw_phase0, cw_psi, J0437-4715_cw_p_dist]>.
This functionality was added in v1.1.0 and may cause post v1.1.0 functionality to break.
This may not cause other errors but it is recommended that you use a custom name for one of the duplicate signals.

Duplicate signal J0437-4715_cw from objects <Enterprise Signal object cw[cw3_cos_gwtheta, cw3_gwphi, cw3_cos_inc, cw3_log10_mc, cw3_log10_fgw, cw3_log10_h, cw3_phase0, cw3_psi, J0437-4715_cw_p_dist]> and <Enterprise Signal object cw[cw_cos_gwtheta, cw_gwphi, cw_cos_inc, cw_log10_mc, cw_log10_fgw, cw_log10_h, cw_phase0, cw_psi, J0437-4715_cw_p_dist]>.
This functionality was added in v1.1.0 and may cause post v1.1.0 functionality to break.


built enterprise PTA N_PSR=10, components=14 in 0.1s
built N_PSR=10, N_CW=2, components=14 in 7.0s
built N_PSR=10, N_CW=4, components=14 in 6.1s
built N_PSR=10, N_CW=6, components=14 in 6.4s
built N_PSR=10, N_CW=8, components=14 in 6.4s
built N_PSR=10, N_CW=12, components=14 in 6.4s
built N_PSR=10, N_CW=16, components=14 in 6.4s
FeatherPulsar.read_feather: cannot find dmx in feather file /home/mattm/projects/HSYMT/data_products/B1855+09.feather.
FeatherPulsar.read_feather: cannot find _pdist in feather file /home/mattm/projects/HSYMT/data_products/B1855+09.feather.
FeatherPulsar.read_feather: cannot find dmx in feather file /home/mattm/projects/HSYMT/data_products/B1937+21.feather.
FeatherPulsar.read_feather: cannot find _pdist in feather file /home/mattm/projects/HSYMT/data_products/B1937+21.feather.
FeatherPulsar.read_feather: cannot find dmx in feather file /home/mattm/projects/HSYMT/data_products/B1953+29.feather.
FeatherPulsar.read_feather: cannot find _pdist in feather file /home

Duplicate signal J0437-4715_cw from objects <Enterprise Signal object cw[cw2_cos_gwtheta, cw2_gwphi, cw2_cos_inc, cw2_log10_mc, cw2_log10_fgw, cw2_log10_h, cw2_phase0, cw2_psi, J0437-4715_cw_p_dist]> and <Enterprise Signal object cw[cw_cos_gwtheta, cw_gwphi, cw_cos_inc, cw_log10_mc, cw_log10_fgw, cw_log10_h, cw_phase0, cw_psi, J0437-4715_cw_p_dist]>.
This functionality was added in v1.1.0 and may cause post v1.1.0 functionality to break.
This may not cause other errors but it is recommended that you use a custom name for one of the duplicate signals.

Duplicate signal J0437-4715_cw from objects <Enterprise Signal object cw[cw3_cos_gwtheta, cw3_gwphi, cw3_cos_inc, cw3_log10_mc, cw3_log10_fgw, cw3_log10_h, cw3_phase0, cw3_psi, J0437-4715_cw_p_dist]> and <Enterprise Signal object cw[cw_cos_gwtheta, cw_gwphi, cw_cos_inc, cw_log10_mc, cw_log10_fgw, cw_log10_h, cw_phase0, cw_psi, J0437-4715_cw_p_dist]>.
This functionality was added in v1.1.0 and may cause post v1.1.0 functionality to break.


built enterprise PTA N_PSR=20, components=14 in 0.3s
built N_PSR=20, N_CW=2, components=14 in 17.6s
built N_PSR=20, N_CW=4, components=14 in 12.2s
built N_PSR=20, N_CW=6, components=14 in 11.6s
built N_PSR=20, N_CW=8, components=14 in 12.0s
built N_PSR=20, N_CW=12, components=14 in 12.1s
built N_PSR=20, N_CW=16, components=14 in 12.1s
FeatherPulsar.read_feather: cannot find dmx in feather file /home/mattm/projects/HSYMT/data_products/B1855+09.feather.
FeatherPulsar.read_feather: cannot find _pdist in feather file /home/mattm/projects/HSYMT/data_products/B1855+09.feather.
FeatherPulsar.read_feather: cannot find dmx in feather file /home/mattm/projects/HSYMT/data_products/B1937+21.feather.
FeatherPulsar.read_feather: cannot find _pdist in feather file /home/mattm/projects/HSYMT/data_products/B1937+21.feather.
FeatherPulsar.read_feather: cannot find dmx in feather file /home/mattm/projects/HSYMT/data_products/B1953+29.feather.
FeatherPulsar.read_feather: cannot find _pdist in feather file

...

Duplicate signal J0437-4715_cw from objects <Enterprise Signal object cw[cw2_cos_gwtheta, cw2_gwphi, cw2_cos_inc, cw2_log10_mc, cw2_log10_fgw, cw2_log10_h, cw2_phase0, cw2_psi, J0437-4715_cw_p_dist]> and <Enterprise Signal object cw[cw_cos_gwtheta, cw_gwphi, cw_cos_inc, cw_log10_mc, cw_log10_fgw, cw_log10_h, cw_phase0, cw_psi, J0437-4715_cw_p_dist]>.
This functionality was added in v1.1.0 and may cause post v1.1.0 functionality to break.
This may not cause other errors but it is recommended that you use a custom name for one of the duplicate signals.

Duplicate signal J0437-4715_cw from objects <Enterprise Signal object cw[cw3_cos_gwtheta, cw3_gwphi, cw3_cos_inc, cw3_log10_mc, cw3_log10_fgw, cw3_log10_h, cw3_phase0, cw3_psi, J0437-4715_cw_p_dist]> and <Enterprise Signal object cw[cw_cos_gwtheta, cw_gwphi, cw_cos_inc, cw_log10_mc, cw_log10_fgw, cw_log10_h, cw_phase0, cw_psi, J0437-4715_cw_p_dist]>.
This functionality was added in v1.1.0 and may cause post v1.1.0 functionality to break.


built enterprise PTA N_PSR=60, components=14 in 2.9s
built N_PSR=60, N_CW=2, components=14 in 59.7s
built N_PSR=60, N_CW=4, components=14 in 48.0s
built N_PSR=60, N_CW=6, components=14 in 48.0s
built N_PSR=60, N_CW=8, components=14 in 48.4s
built N_PSR=60, N_CW=12, components=14 in 48.5s
built N_PSR=60, N_CW=16, components=14 in 47.0s
FeatherPulsar.read_feather: cannot find dmx in feather file /home/mattm/projects/HSYMT/data_products/B1855+09.feather.
FeatherPulsar.read_feather: cannot find _pdist in feather file /home/mattm/projects/HSYMT/data_products/B1855+09.feather.
FeatherPulsar.read_feather: cannot find dmx in feather file /home/mattm/projects/HSYMT/data_products/B1937+21.feather.
FeatherPulsar.read_feather: cannot find _pdist in feather file /home/mattm/projects/HSYMT/data_products/B1937+21.feather.
FeatherPulsar.read_feather: cannot find dmx in feather file /home/mattm/projects/HSYMT/data_products/B1953+29.feather.
FeatherPulsar.read_feather: cannot find _pdist in feather file

Duplicate signal J0437-4715_cw from objects <Enterprise Signal object cw[cw2_cos_gwtheta, cw2_gwphi, cw2_cos_inc, cw2_log10_mc, cw2_log10_fgw, cw2_log10_h, cw2_phase0, cw2_psi, J0437-4715_cw_p_dist]> and <Enterprise Signal object cw[cw_cos_gwtheta, cw_gwphi, cw_cos_inc, cw_log10_mc, cw_log10_fgw, cw_log10_h, cw_phase0, cw_psi, J0437-4715_cw_p_dist]>.
This functionality was added in v1.1.0 and may cause post v1.1.0 functionality to break.
This may not cause other errors but it is recommended that you use a custom name for one of the duplicate signals.

Duplicate signal J0437-4715_cw from objects <Enterprise Signal object cw[cw3_cos_gwtheta, cw3_gwphi, cw3_cos_inc, cw3_log10_mc, cw3_log10_fgw, cw3_log10_h, cw3_phase0, cw3_psi, J0437-4715_cw_p_dist]> and <Enterprise Signal object cw[cw_cos_gwtheta, cw_gwphi, cw_cos_inc, cw_log10_mc, cw_log10_fgw, cw_log10_h, cw_phase0, cw_psi, J0437-4715_cw_p_dist]>.
This functionality was added in v1.1.0 and may cause post v1.1.0 functionality to break.


built enterprise PTA N_PSR=80, components=14 in 5.3s
built N_PSR=80, N_CW=2, components=14 in 94.6s
built N_PSR=80, N_CW=4, components=14 in 77.9s
built N_PSR=80, N_CW=6, components=14 in 80.7s
built N_PSR=80, N_CW=8, components=14 in 80.8s
built N_PSR=80, N_CW=12, components=14 in 81.2s
built N_PSR=80, N_CW=16, components=14 in 78.9s


,N_PSR,N_CW,components,seconds
0,10,2,14,6.968786
1,10,4,14,6.144221
2,10,6,14,6.364213
3,10,8,14,6.352989
4,10,12,14,6.366860
5,10,16,14,6.389123
6,20,2,14,17.552400
7,20,4,14,12.249073
8,20,6,14,11.648913
9,20,8,14,12.008823


## Validate Components Reduction

In [7]:
validation_df = pd.DataFrame()

if VALIDATE_COMPONENTS:
    print("VALIDATE COMPONENTS REDUCTION")
    rows = []
    for comp in [14, 30]:
        case = build_case(VALIDATION_N_PSR, VALIDATION_N_CW, comp)
        idxs = diagnostic_indices(case["sigmas"], case["mode_spacings"])
        contrasts = []
        for i in idxs:
            m = scan_metrics(case, i, case["truth_dist"], scan_points=TIER1_SCAN_POINTS)
            contrasts.append(m["contrast"])
        rows.append(dict(
            components=comp,
            N_PSR=VALIDATION_N_PSR,
            N_CW=VALIDATION_N_CW,
            median_contrast_truth=float(np.nanmedian(contrasts)),
            diagnostic_indices=idxs,
        ))
        print(f"  components={comp}: median truth-context contrast={np.nanmedian(contrasts):.3f}")
    validation_df = pd.DataFrame(rows)
    display(validation_df)
else:
    print("Component validation skipped")

VALIDATE COMPONENTS REDUCTION
  components=14: median truth-context contrast=4245.644
FeatherPulsar.read_feather: cannot find dmx in feather file /home/mattm/projects/HSYMT/data_products/B1855+09.feather.
FeatherPulsar.read_feather: cannot find _pdist in feather file /home/mattm/projects/HSYMT/data_products/B1855+09.feather.
FeatherPulsar.read_feather: cannot find dmx in feather file /home/mattm/projects/HSYMT/data_products/B1937+21.feather.
FeatherPulsar.read_feather: cannot find _pdist in feather file /home/mattm/projects/HSYMT/data_products/B1937+21.feather.
FeatherPulsar.read_feather: cannot find dmx in feather file /home/mattm/projects/HSYMT/data_products/B1953+29.feather.
FeatherPulsar.read_feather: cannot find _pdist in feather file /home/mattm/projects/HSYMT/data_products/B1953+29.feather.
FeatherPulsar.read_feather: cannot find dmx in feather file /home/mattm/projects/HSYMT/data_products/J0023+0923.feather.
FeatherPulsar.read_feather: cannot find _pdist in feather file /home/m

Duplicate signal J0437-4715_cw from objects <Enterprise Signal object cw[cw2_cos_gwtheta, cw2_gwphi, cw2_cos_inc, cw2_log10_mc, cw2_log10_fgw, cw2_log10_h, cw2_phase0, cw2_psi, J0437-4715_cw_p_dist]> and <Enterprise Signal object cw[cw_cos_gwtheta, cw_gwphi, cw_cos_inc, cw_log10_mc, cw_log10_fgw, cw_log10_h, cw_phase0, cw_psi, J0437-4715_cw_p_dist]>.
This functionality was added in v1.1.0 and may cause post v1.1.0 functionality to break.
This may not cause other errors but it is recommended that you use a custom name for one of the duplicate signals.

Duplicate signal J0437-4715_cw from objects <Enterprise Signal object cw[cw3_cos_gwtheta, cw3_gwphi, cw3_cos_inc, cw3_log10_mc, cw3_log10_fgw, cw3_log10_h, cw3_phase0, cw3_psi, J0437-4715_cw_p_dist]> and <Enterprise Signal object cw[cw_cos_gwtheta, cw_gwphi, cw_cos_inc, cw_log10_mc, cw_log10_fgw, cw_log10_h, cw_phase0, cw_psi, J0437-4715_cw_p_dist]>.
This functionality was added in v1.1.0 and may cause post v1.1.0 functionality to break.


built enterprise PTA N_PSR=20, components=30 in 0.3s
built N_PSR=20, N_CW=8, components=30 in 17.1s
  components=30: median truth-context contrast=4034.847


,components,N_PSR,N_CW,median_contrast_truth,diagnostic_indices
0,14,20,8,4245.644317,"[1, 5, 10, 14, 19]"
1,30,20,8,4034.847142,"[1, 5, 10, 14, 19]"


## Tier 0 - Global Signal Strength

In [8]:
tier0_df = pd.DataFrame()

if RUN_TIER_0:
    print("TIER 0 - GLOBAL SIGNAL STRENGTH")
    rows = []
    for n_psr, n_cw in itertools.product(N_PSR_VALUES, N_CW_VALUES):
        t0 = time.time()
        case = build_case(n_psr, n_cw, COMPONENTS)
        truth_lnL = float(case["logl_fn"](jnp.asarray(case["truth_values"])))
        prior_lnL = float(case["logl_fn"](jnp.asarray(case["prior_values"])))
        delta_global = truth_lnL - prior_lnL
        try:
            opt = make_distance_optimizer(
                case["logl_fn"], case["param_keys"], case["base_values"], case["disco_psrs"],
                prior_means=case["prior_mean"], prior_sigmas=case["sigmas"],
                n_sigma=MODE_PRIOR_SIGMA_WIDTH, objective="lnL", maxiter=TIER0_OPT_MAXITER,
                factr=OPT_FACTR,
            )[0]
        except TypeError:
            opt = make_distance_optimizer(
                case["logl_fn"], case["param_keys"], case["base_values"], case["disco_psrs"],
                prior_means=case["prior_mean"], prior_sigmas=case["sigmas"],
                n_sigma=MODE_PRIOR_SIGMA_WIDTH, objective="lnL", maxiter=TIER0_OPT_MAXITER,
            )[0]
        d_opt, lnL_opt, _, info = opt(case["truth_dist"])
        truth_start_gain = float(lnL_opt - truth_lnL)
        rows.append(dict(
            N_PSR=n_psr, N_CW=n_cw, truth_lnL=truth_lnL, prior_lnL=prior_lnL,
            delta_global=delta_global, truth_start_gain=truth_start_gain,
            nit=info.get("nit"), seconds=time.time() - t0,
        ))
        print(f"  N_PSR={n_psr:2d} N_CW={n_cw:2d}: delta={delta_global:.3f}, truth_gain={truth_start_gain:.3f}")
    tier0_df = pd.DataFrame(rows)
    display(tier0_df)
else:
    print("Tier 0 skipped")

TIER 0 - GLOBAL SIGNAL STRENGTH
  N_PSR=10 N_CW= 2: delta=213689.569, truth_gain=3.757
  N_PSR=10 N_CW= 4: delta=592618.966, truth_gain=5.033
  N_PSR=10 N_CW= 6: delta=729481.920, truth_gain=12.084
  N_PSR=10 N_CW= 8: delta=853487.366, truth_gain=12.194
  N_PSR=10 N_CW=12: delta=1505314.774, truth_gain=15.463
  N_PSR=10 N_CW=16: delta=1779812.478, truth_gain=17.685
  N_PSR=20 N_CW= 2: delta=338739.344, truth_gain=15.558
  N_PSR=20 N_CW= 4: delta=563842.895, truth_gain=10.696
  N_PSR=20 N_CW= 6: delta=1038315.271, truth_gain=4.423
  N_PSR=20 N_CW= 8: delta=1249605.523, truth_gain=6.884
  N_PSR=20 N_CW=12: delta=2379259.749, truth_gain=6.311
  N_PSR=20 N_CW=16: delta=2882868.444, truth_gain=9.690
  N_PSR=40 N_CW= 2: delta=1164167.094, truth_gain=32.428
  N_PSR=40 N_CW= 4: delta=1844751.503, truth_gain=29.294
  N_PSR=40 N_CW= 6: delta=2081381.160, truth_gain=25.816
  N_PSR=40 N_CW= 8: delta=2301003.100, truth_gain=24.930
  N_PSR=40 N_CW=12: delta=3964039.959, truth_gain=60.187
  N_PSR=40 

,N_PSR,N_CW,truth_lnL,prior_lnL,delta_global,truth_start_gain,nit,seconds
0,10,2,64978.489214,-1.487111e+05,2.136896e+05,3.756984,14,15.532443
1,10,4,64978.489214,-5.276405e+05,5.926190e+05,5.033326,13,10.699279
2,10,6,64978.489214,-6.645034e+05,7.294819e+05,12.083853,11,10.960638
3,10,8,64978.489214,-7.885089e+05,8.534874e+05,12.194180,11,11.180984
4,10,12,64978.489214,-1.440336e+06,1.505315e+06,15.462785,20,11.331051
5,10,16,64978.489214,-1.714834e+06,1.779812e+06,17.684824,20,14.538211
6,20,2,99917.936887,-2.388214e+05,3.387393e+05,15.558400,20,26.521332
7,20,4,99917.936887,-4.639250e+05,5.638429e+05,10.696254,20,20.527146
8,20,6,99917.936887,-9.383973e+05,1.038315e+06,4.422524,20,20.776734
9,20,8,99917.936887,-1.149688e+06,1.249606e+06,6.883959,20,20.731073


## Tier 1 - Mode Contrast Survey

In [ ]:
tier1_detail_df = pd.DataFrame()
tier1_df = pd.DataFrame()

if RUN_TIER_1:
    print("TIER 1 - MODE CONTRAST SURVEY")
    detail_rows = []
    summary_rows = []
    for n_psr, n_cw in itertools.product(N_PSR_VALUES, N_CW_VALUES):
        t0 = time.time()
        case = build_case(n_psr, n_cw, COMPONENTS)
        idxs = diagnostic_indices(case["sigmas"], case["mode_spacings"])
        truth_contrasts = []
        prior_contrasts = []
        truth_globals = []
        degenerates = []
        for i in idxs:
            mt = scan_metrics(case, i, case["truth_dist"], scan_points=TIER1_SCAN_POINTS)
            mp = scan_metrics(case, i, case["prior_mean"], scan_points=TIER1_SCAN_POINTS)
            hardness = case["sigmas"][i] / case["mode_spacings"][i]
            truth_contrasts.append(mt["contrast"])
            prior_contrasts.append(mp["contrast"])
            truth_globals.append(mt["truth_is_global"])
            degenerates.append(mt["n_degenerate"])
            detail_rows.append(dict(
                N_PSR=n_psr, N_CW=n_cw, idx=i, pulsar=case["disco_psrs"][i].name,
                sigma_over_mode=float(hardness),
                contrast_truth=mt["contrast"], contrast_prior=mp["contrast"],
                truth_is_global=mt["truth_is_global"], n_degenerate=mt["n_degenerate"],
            ))
        row = dict(
            N_PSR=n_psr, N_CW=n_cw,
            median_contrast_truth=float(np.nanmedian(truth_contrasts)),
            median_contrast_prior=float(np.nanmedian(prior_contrasts)),
            frac_truth_is_global=float(np.mean(truth_globals)),
            avg_n_degenerate=float(np.mean(degenerates)),
            diagnostic_indices=idxs,
            seconds=time.time() - t0,
        )
        summary_rows.append(row)
        print(
            f"  N_PSR={n_psr:2d} N_CW={n_cw:2d}: "
            f"med_contrast={row['median_contrast_truth']:.3f}, "
            f"truth_global={row['frac_truth_is_global']:.2f}"
        )
    tier1_detail_df = pd.DataFrame(detail_rows)
    tier1_df = pd.DataFrame(summary_rows)
    display(tier1_df)
else:
    print("Tier 1 skipped")

TIER 1 - MODE CONTRAST SURVEY
  N_PSR=10 N_CW= 2: med_contrast=3244.058, truth_global=0.80
  N_PSR=10 N_CW= 4: med_contrast=9778.271, truth_global=1.00
  N_PSR=10 N_CW= 6: med_contrast=10080.312, truth_global=1.00
  N_PSR=10 N_CW= 8: med_contrast=10874.091, truth_global=1.00
  N_PSR=10 N_CW=12: med_contrast=5657.030, truth_global=1.00
  N_PSR=10 N_CW=16: med_contrast=8783.382, truth_global=1.00
  N_PSR=20 N_CW= 2: med_contrast=7169.664, truth_global=0.80
  N_PSR=20 N_CW= 4: med_contrast=28534.564, truth_global=1.00
  N_PSR=20 N_CW= 6: med_contrast=11032.730, truth_global=1.00
  N_PSR=20 N_CW= 8: med_contrast=4245.644, truth_global=1.00
  N_PSR=20 N_CW=12: med_contrast=10018.477, truth_global=1.00
  N_PSR=20 N_CW=16: med_contrast=16544.999, truth_global=1.00
  N_PSR=40 N_CW= 2: med_contrast=2683.353, truth_global=0.60
  N_PSR=40 N_CW= 4: med_contrast=3041.039, truth_global=1.00
  N_PSR=40 N_CW= 6: med_contrast=3141.431, truth_global=1.00
  N_PSR=40 N_CW= 8: med_contrast=4110.811, truth_

## Tier 2 - Fast Coordinate Descent Recovery

In [ ]:
tier2_df = pd.DataFrame()


def fast_coordinate_descent(case, n_sweeps=COORD_DESCENT_SWEEPS, scan_points=COORD_DESCENT_SCAN_POINTS):
    order = np.argsort(case["sigmas"] / case["mode_spacings"])
    current = case["prior_mean"].copy()
    rows = []
    for sweep in range(n_sweeps):
        for i in order:
            ctx = set_distances(case["base_values"], case["param_keys"], case["disco_psrs"], current)
            lo = max(0.01, case["prior_mean"][i] - MODE_PRIOR_SIGMA_WIDTH * case["sigmas"][i])
            hi = case["prior_mean"][i] + MODE_PRIOR_SIGMA_WIDTH * case["sigmas"][i]
            vals, lls = scan_pulsar_distance(
                case["logl_fn"], ctx, case["param_keys"], distance_key(case["disco_psrs"][i]),
                lo, hi, n_points=scan_points, required_points=[case["truth_dist"][i], case["prior_mean"][i]],
                n_components=case["components"],
            )
            current[i] = float(np.asarray(vals)[int(np.nanargmax(lls))])
        sc = score_recovery(current, case["truth_dist"], case["mode_spacings"])
        rows.append(dict(sweep=sweep + 1, **sc))
        print(f"    sweep {sweep + 1}: {sc['n_recovered']}/{case['n_psr']}")
    return current, pd.DataFrame(rows)


if RUN_TIER_2:
    print("TIER 2 - FAST COORDINATE DESCENT")
    rows = []
    if tier1_df.empty:
        print("Tier 1 missing; running Tier 2 for all configs.")
        candidates = [(n, c) for n in N_PSR_VALUES for c in N_CW_VALUES]
    else:
        passed = tier1_df[tier1_df["median_contrast_truth"] > TIER_2_CONTRAST_THRESHOLD]
        candidates = list(zip(passed["N_PSR"].astype(int), passed["N_CW"].astype(int)))
        print(f"Tier 2 configs passing contrast>{TIER_2_CONTRAST_THRESHOLD}: {len(candidates)}")
    for n_psr, n_cw in candidates:
        t0 = time.time()
        case = build_case(n_psr, n_cw, COMPONENTS)
        print(f"  N_PSR={n_psr:2d} N_CW={n_cw:2d}")
        d_cd, hist = fast_coordinate_descent(case)
        rec = score_recovery(d_cd, case["truth_dist"], case["mode_spacings"])
        row = dict(N_PSR=n_psr, N_CW=n_cw, total_seconds=time.time() - t0, **rec)
        for _, h in hist.iterrows():
            row[f"frac_recovered_sweep{int(h['sweep'])}"] = h["frac_recovered"]
        rows.append(row)
    tier2_df = pd.DataFrame(rows)
    display(tier2_df)
else:
    print("Tier 2 skipped")

## Scaling Survey Results And Plots

In [ ]:
print("SCALING SURVEY RESULTS")
print("======================")

result = pd.DataFrame([(n, c) for n in N_PSR_VALUES for c in N_CW_VALUES], columns=["N_PSR", "N_CW"])
if not tier0_df.empty:
    result = result.merge(tier0_df[["N_PSR", "N_CW", "delta_global", "truth_start_gain"]], on=["N_PSR", "N_CW"], how="left")
if not tier1_df.empty:
    result = result.merge(tier1_df[["N_PSR", "N_CW", "median_contrast_truth", "median_contrast_prior", "frac_truth_is_global", "avg_n_degenerate"]], on=["N_PSR", "N_CW"], how="left")
if not tier2_df.empty:
    result = result.merge(tier2_df[["N_PSR", "N_CW", "frac_recovered", "total_seconds"]], on=["N_PSR", "N_CW"], how="left")

display(result.sort_values(["N_PSR", "N_CW"]))

if MAKE_PLOTS:
    fig, axes = plt.subplots(1, 3, figsize=(17, 4.5), constrained_layout=True)

    def heat(ax, data, value, title, vmin=None, vmax=None, cmap="viridis"):
        piv = data.pivot(index="N_PSR", columns="N_CW", values=value).reindex(index=N_PSR_VALUES, columns=N_CW_VALUES)
        im = ax.imshow(piv.to_numpy(float), origin="lower", aspect="auto", vmin=vmin, vmax=vmax, cmap=cmap)
        ax.set_xticks(range(len(N_CW_VALUES)), N_CW_VALUES)
        ax.set_yticks(range(len(N_PSR_VALUES)), N_PSR_VALUES)
        ax.set_xlabel("N_CW")
        ax.set_ylabel("N_PSR")
        ax.set_title(title)
        for yi, n in enumerate(N_PSR_VALUES):
            for xi, c in enumerate(N_CW_VALUES):
                val = piv.loc[n, c]
                if pd.notna(val):
                    ax.text(xi, yi, f"{val:.1f}", ha="center", va="center", color="white" if val > (np.nanmax(piv.to_numpy(float)) / 2) else "black", fontsize=8)
        fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

    if "median_contrast_truth" in result:
        heat(axes[0], result, "median_contrast_truth", "Median Mode Contrast", vmin=0, vmax=max(50, np.nanmax(result["median_contrast_truth"])), cmap="magma")
    else:
        axes[0].set_axis_off()

    if "frac_recovered" in result:
        heat(axes[1], result, "frac_recovered", "Coordinate Descent Recovery", vmin=0, vmax=1, cmap="viridis")
    else:
        axes[1].set_axis_off()

    if "frac_recovered" in result:
        for n_psr in N_PSR_VALUES:
            sub = result[result["N_PSR"] == n_psr].sort_values("N_CW")
            axes[2].plot(sub["N_CW"], sub["frac_recovered"], marker="o", label=f"N_PSR={n_psr}")
        axes[2].set_ylim(-0.05, 1.05)
        axes[2].set_xlabel("N_CW")
        axes[2].set_ylabel("Recovery Fraction")
        axes[2].set_title("Recovery vs N_CW")
        axes[2].legend()
        axes[2].grid(True, alpha=0.3)
    else:
        axes[2].set_axis_off()

    plt.show()
else:
    print("Plots skipped")